# lenia-vlm on Colab: MaCE-Lenia runs

GPU runs for the live pipeline (see README.md). Results go to your Drive folder `lenia-runs/` and survive
disconnects; every run resumes if you re-run sections 1-2 and then the same command with the same `--out`.
The earlier 19-section notebook (classic Lenia, distillation, hill climber, MAP-Elites v1-v4) is in
`archive/colab/lenia_colab.ipynb`.

## 1. GPU and Drive

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> GPU"
import os
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/lenia-runs'      # every result goes here and survives disconnects
os.makedirs(RUNS, exist_ok=True)
print('results folder:', RUNS)

## 2. Code, dependencies, judge keys

In [ ]:
import os, subprocess
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

REPO, DIR = 'github.com/feli10/lenia-vlm.git', '/content/lenia-vlm'
tok = secret('GITHUB_TOKEN')
url = f'https://x-access-token:{tok}@{REPO}' if tok else f'https://{REPO}'
if not os.path.isdir(DIR):
    r = subprocess.run(['git', 'clone', '-q', url, DIR], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', DIR, 'pull', '-q', url, 'main'], capture_output=True, text=True)
print('git:', 'ok' if r.returncode == 0 else r.stderr.replace(tok or '\0', '***'))
subprocess.run(['git', '-C', DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'])   # don't keep the token in .git
os.chdir(DIR)
!git log --oneline -1
!pip install -q -e .

In [ ]:
# API keys for the judges, if you added them as Colab secrets
keys = {k: secret(k) for k in ('OPENROUTER_API_KEY', 'TYPESAFE_API_KEY', 'GOOGLE_CLOUD_PROJECT')}
with open('.env', 'w') as f:
    for k, v in keys.items():
        if v:
            f.write(f'{k}={v}\n')
print('judge keys present:', [k for k, v in keys.items() if v] or 'none (fine for simulation-only runs)')

## 3. Checks

The test suite, offline: the golden test (prompts, pictures, cache keys, mutation and simulation unchanged, so
old caches and runs stay valid) and the PyTorch-vs-numpy parity tests. Do not use the GPU for runs unless these
pass.

In [ ]:
!pip install -q pytest && python -m pytest -q tests

## 4. Base search: MAP-Elites + surprise archive

The command that produced `mace-qd5-sarch-s0`, the seed 0 base run the stage-1 core test reuses (4 islands,
bodies x form x motion, complexity entry duel, surprise archive). About 2.3 h and ~$3 for 100 iterations.
Kept for reference; the seed 1 base run for stage 1 (shorter, with the record logging) is added here when it
is built.

In [ ]:
# reference: seed 0 (already done)
# !cd /content/lenia-vlm && python scripts/mace_qd.py run --backend auto --islands 4 --axes bodies,form,motion --body-edges 3,9 --motion-edges 0.75,1.0,1.25 --iterations 100 --sarch --seed 0 --out {RUNS}/mace-qd5-sarch-s0

## 5. Stage 1: expeditions (seed 0)

Each arm runs one 10-step expedition (32 gentle mutants a step, judged against the trigger's start, comma selection: expedition v3) from each of the 12 triggers in `stage1-s0/triggers.json`.
Copy the Mac's `runs/stage1-s0` folder to Drive `lenia-runs/stage1-s0` first (triggers, pictures, the shared judge and
VLM caches). Run the arms in order: **L** (then `check`: gate 0c, the floor), then **N**, **NR**, **NS**. NR and NS
reuse N's first 5 steps from the caches and ask Gemini Pro for one review per trigger, which needs the Gemini cell
below (Vertex AI, billed to your Google Cloud project; add `GOOGLE_CLOUD_PROJECT` as a Colab secret). Every command
resumes where it stopped.

Results go to `stage1-s0/exp-v3/<arm>/` (v2 in `exp-v2/`, v1 in `arms/`, kept as the record). The simulation cache lives on local disk (`--evals /content/evals`); accepted worlds are copied to Drive.

In [ ]:
# Gemini on Vertex (arms NR and NS; arm N's names are already cached): sign in once per session.
# No google-genai install: lenia.describe falls back to Vertex's REST API with Colab's own google-auth
# (installing the SDK upgrades the google-auth that Colab pins).
from google.colab import auth
auth.authenticate_user()

In [ ]:
S1, BASE = f'{RUNS}/stage1-s0', f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/stage1.py expedition --arm L --run {BASE} --out {S1} --backend auto --evals /content/evals

In [ ]:
# held-out Luna, end vs start (gate 0c for arm L); no GPU needed, can also run on the Mac
!cd /content/lenia-vlm && python scripts/stage1.py check --arm L --run {BASE} --out {S1}

In [ ]:
# after gate 0c passes and Felix has looked at exp-v3/L/check.png:
# !cd /content/lenia-vlm && python scripts/stage1.py expedition --arm N  --run {BASE} --out {S1} --backend auto --evals /content/evals
# !cd /content/lenia-vlm && python scripts/stage1.py expedition --arm NR --run {BASE} --out {S1} --backend auto --evals /content/evals
# !cd /content/lenia-vlm && python scripts/stage1.py expedition --arm NS --run {BASE} --out {S1} --backend auto --evals /content/evals
# !cd /content/lenia-vlm && for A in N NR NS; do python scripts/stage1.py check --arm $A --run {BASE} --out {S1}; done

## 6. CLIP targeted-search bench (no API calls)

Which search settings actually steer a world toward a text target? CLIP scores each child (final snapshot vs the
prompt), so it costs nothing but GPU time (~1-1.5 h on a T4 for the default 6 configurations x 4 targets x 2 starts
x 12 steps x 16 children). Compares world inheritance vs regrowth from noise, mutation strength, sep-CMA-ES (ASAL's
optimiser) and a best-of-N local control. Resumes after a disconnect. See `scripts/clipbench.py`.

In [ ]:
BASE = f'{RUNS}/mace-qd5-sarch-s0'      # the base run (start worlds); defined here so this section runs on its own
!pip install -q cma
!cd /content/lenia-vlm && python scripts/clipbench.py run --run {BASE} --out {RUNS}/clipbench --backend auto

In [ ]:
# Round 1 before/after films (one GIF per target; rows = start worlds, columns = start | best per setting). A few min on GPU.
BASE = f'{RUNS}/mace-qd5-sarch-s0'      # the start worlds (defined in every cell, so each runs on its own)
!cd /content/lenia-vlm && python scripts/clipbench.py gifs --run {BASE} --out {RUNS}/clipbench --backend auto
import glob
from IPython.display import Image as GIF, display
for f in sorted(glob.glob(f'{RUNS}/clipbench/gifs/*.gif')):
    print(f); display(GIF(filename=f))

In [ ]:
# Round 2: can Jev steer? Jev selects (goal score vs the start), CLIP scores every child, held out. ~$0.60 of Jev, ~1 h (20 steps: CLIP's climb plateaus ~22).
# Needs .env with the TypeSafe key in /content/lenia-vlm (as for the expeditions). Resumable: re-run the cell if it stops.
BASE = f'{RUNS}/mace-qd5-sarch-s0'      # the start worlds (defined in every cell, so each runs on its own)
!cd /content/lenia-vlm && LENIA_VLM_CACHE_ONLY=1 python scripts/clipbench.py run --run {BASE} --out {RUNS}/clipbench-jev --backend auto --configs es-regrow-1,jev-regrow-1,local-1 --steps 20
!cd /content/lenia-vlm && python scripts/clipbench.py report --out {RUNS}/clipbench-jev

In [ ]:
# Round 2 films: start | CLIP-steered best | Jev-steered best, per target (rows = starts). Also re-prints the report
# with Jev's own scores per step. A few min on GPU, no API calls.
BASE = f'{RUNS}/mace-qd5-sarch-s0'
!cd /content/lenia-vlm && python scripts/clipbench.py report --out {RUNS}/clipbench-jev | grep -A3 'Jev-steered'
!cd /content/lenia-vlm && python scripts/clipbench.py gifs --run {BASE} --out {RUNS}/clipbench-jev --backend auto --configs es-regrow-1,jev-regrow-1
import glob
from IPython.display import Image as GIF, display
for f in sorted(glob.glob(f'{RUNS}/clipbench-jev/gifs/*.gif')):
    print(f); display(GIF(filename=f))

In [ ]:
# Long run: does the climb keep going past 12 steps, and can it steer far (pizza)? CLIP only, free, ~35 min. Resumable.
BASE = f'{RUNS}/mace-qd5-sarch-s0'      # the start worlds (defined in every cell, so each runs on its own)
!cd /content/lenia-vlm && python scripts/clipbench.py run --run {BASE} --out {RUNS}/clipbench-long --backend auto \
    --configs es-regrow-1 --steps 40 --prompts "a worm|a ring|a starfish|a pepperoni pizza"
!cd /content/lenia-vlm && python scripts/clipbench.py gifs --run {BASE} --out {RUNS}/clipbench-long --backend auto --configs es-regrow-1
import glob
from IPython.display import Image as GIF, display
for f in sorted(glob.glob(f'{RUNS}/clipbench-long/gifs/*.gif')):
    print(f); display(GIF(filename=f))

In [ ]:
# Is CLIP's steering target-specific, or just making worlds more CLIP-friendly in general? ~1 min, free.
!cd /content/lenia-vlm && python scripts/clipbench.py cross --out {RUNS}/clipbench-long
!cd /content/lenia-vlm && python scripts/clipbench.py cross --out {RUNS}/clipbench | tail -8

In [ ]:
# Round 3: sharpen Jev's choice and fight the plateau. Each config differs from jev-regrow-1 (round 2's Jev setting) in one
# thing: playoff (top 4 meet head-to-head), clipfilter (48 children, CLIP shortlists 16, Jev picks), sched (mutation
# 2.0 -> 0.5). Same targets, starts and mutants as round 2. jev-regrow-1 reuses round 2's cached Jev answers.
# GIFs are BLIND: columns lettered and shuffled per target, key in gifs/key.json (rate before opening it).
# ~$1.9 of new Jev (cap $3.5 includes the copied round-2 answers); ~2.5-3 h on a T4. Resumable: re-run the cell.
import os, shutil
BASE = f'{RUNS}/mace-qd5-sarch-s0'
R3 = f'{RUNS}/clipbench-r3'
os.makedirs(R3, exist_ok=True)
if os.path.exists(f'{RUNS}/clipbench-jev/judge_cache.sqlite') and not os.path.exists(f'{R3}/judge_cache.sqlite'):
    shutil.copy(f'{RUNS}/clipbench-jev/judge_cache.sqlite', f'{R3}/judge_cache.sqlite')
!cd /content/lenia-vlm && LENIA_VLM_CACHE_ONLY=1 python scripts/clipbench.py run --run {BASE} --out {R3} --backend auto \
    --configs jev-regrow-1,jev-playoff-1,jev-clipfilter-1,jev-sched --steps 20 --max-usd 3.5
!cd /content/lenia-vlm && python scripts/clipbench.py gifs --run {BASE} --out {R3} --backend auto --blind \
    --configs jev-regrow-1,jev-playoff-1,jev-clipfilter-1,jev-sched
import glob
from IPython.display import Image as GIF, display
for f in sorted(glob.glob(f'{R3}/gifs/*_blind.gif')):
    print(f); display(GIF(filename=f))

## Notes

- The free tier disconnects idle sessions and caps sessions at about 12 hours. After a disconnect the machine
  is wiped, your Drive is not: re-run sections 1-2, then the same command with the same `--out`.
- Judge calls made here are cached in the run folder (`judge_cache.sqlite`), so a resumed run does not pay twice.
- Claude can read your Drive folder: say when a run has finished.